### Set autoreloading
This extension will automatically update with any changes to packages in real time

In [1]:
%load_ext autoreload
%autoreload 2

### Import packages
We'll need the `nugraph` and `pynuml` packages imported in order to plot, and `torch` for some tensor operations later on

In [2]:
import os
import nugraph as ng
import pynuml
import torch
print(ng.__file__)

/home/twalton/.conda/envs/NugraphBase/lib/python3.10/site-packages/torch/cuda/__init__.py:64: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/nugraph/nugraph/__init__.py


### Set model and data to use

This allows the user to switch out different model architectures and datasets

In [3]:
Data = ng.data.NuGraphDataModule
Model = ng.models.NuGraph3

### Configure data module
Declare a data module. If you're working on a standard cluster, the data file location should be configured automatically. If not, you'll need to configure it manually.

In [6]:
source_filename = "/scratch/7DayLifetime/cerati/concat-final-makeup.half.gnn.h5"
target_filename = "/scratch/7DayLifetime/cerati/icarus-numi-1d.gnn.h5"
data_batch_size = 64*2 
nudata = Data(model=Model, batch_size=data_batch_size, data_source_path=source_filename, data_target_path=target_filename)

print( "The Nugraph Data (nudata) type is", type(nudata) )

The Nugraph Data (nudata) type is <class 'nugraph.data.data_module.NuGraphDataModule'>


### Configure network
In order to test a trained model, we instantiate it using a checkpoint file. These are produced during training, so if you've trained a model, there should be an associated checkpoint in your output directory that you can pass here.

In [9]:
os.environ["NUGRAPH_TOP_DIR"]="/home/twalton/NuGraphLogs/NuGraph-DA-OFF/EPOCH70-WARMUP70/MicroBoone-Icarus-Large-2026-08-13-18/"
ckpt = os.path.expandvars("$NUGRAPH_TOP_DIR/version_0/checkpoints/epoch=37-step=8892.ckpt")

"""
ckpt_dict = torch.load(ckpt, map_location="cpu")
print("=== Top‑level keys in the checkpoint file ===")
for k in ckpt_dict.keys():
    print("-", k)
print( "\n" )
"""

model = Model.load_from_checkpoint(ckpt, map_location="cpu")
"""
print("=== Parameter / buffer names that the model now holds ===")
for name in model.state_dict().keys():
    print("-", name)
print( "\n" )
print( model.hparams )
"""
model.freeze()

NuGraph3(
  (encoder): Encoder(
    (input_norms): ModuleDict(
      (source): InputNorm(
        (norm): ParameterDict(
            (count): Parameter containing: [torch.LongTensor of size 1]
            (mean): Parameter containing: [torch.FloatTensor of size 5]
            (var): Parameter containing: [torch.FloatTensor of size 5]
        )
      )
      (target): InputNorm(
        (norm): ParameterDict(
            (count): Parameter containing: [torch.LongTensor of size 1]
            (mean): Parameter containing: [torch.FloatTensor of size 5]
            (var): Parameter containing: [torch.FloatTensor of size 5]
        )
      )
    )
    (planar_net): Linear(in_features=5, out_features=128, bias=True)
    (beta_net): Sequential(
      (0): Linear(in_features=5, out_features=1, bias=True)
      (1): Sigmoid()
    )
    (coord_net): Sequential(
      (0): Linear(in_features=5, out_features=32, bias=True)
      (1): Mish()
    )
  )
  (core_net): NuGraphCore(
    (plane_net): NuG

### Configure plotting utility
Instantiate the **pynuml** utility for plotting graph objects, which will do the heavy lifting for us here!

In [13]:
classes = nudata.event_classes+['nan'] 
plot    = pynuml.plot.GraphPlot(planes=nudata.planes,classes=classes) #semantic_classes) #nudata.event_classes)
print( "The nudata.event_classes are", classes )

The nudata.event_classes are ['cc_nue', 'cc_numu', 'cc_nutau', 'nc', 'nan']


### Plot ground truth labels
#### Iterable dataset
First, we define an iterator over the test dataset:

In [17]:
test_iter = iter(nudata.combined_test)
print( "The test_iter type is", type(test_iter), type(nudata.combined_test), nudata.combined_test )

The test_iter type is <class 'generator'> <class 'nugraph.data.dataset.NuGraphCombinedDataset'> NuGraphCombinedDataset(1663)


### Retrieve the next graph
This block retrieves a graph from the testing dataset and passes it through the trained model. 
Since we defined `test_iter` as an iterator over the dataset, the following block can be executed multiple times; 
each time, it steps to the next graph in the dataset.

In [20]:
data = next(test_iter)
batchA,batchB = data

metadata = names = []
for batch in [batchA,batchB] :
    metadata.append( batch['metadata'] )
    names.append( f'r{metadata[-1].run}_sr{metadata[-1].subrun}_e{metadata[-1].event}' )
    print( "Batch is", type(batch), len(batch), names[-1] )

model(data=[batchA,batchB])

/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y', 'y_vtx'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):
/home/twalton/NuGraphGPUWorkspace/NuGraphMain/nugraph/pynuml/pynuml/data/nugraph_data.py:108: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y'}'. Please explicitly set 'num_nodes' as an attribute of 'data[evt]' to suppress this warning
  if n.num_nodes is not None and not hasattr(n, "x"):


Batch is <class 'pynuml.data.nugraph_data.NuGraphData'> 12 r6426_sr77_e3867
Batch is <class 'pynuml.data.nugraph_data.NuGraphData'> 14 r569_sr3_e5
self.da_loss_fnc_name dann


(tensor(-1.8102), {})

#### Plot a single graph
We can now use pynuml's plotting utilities to plot the graph as a figure. 
Each time you call the above block to retrieve a new graph, you can then re-execute 
the plotting blocks to re-plot with the new graph.

In [32]:
figures = []
for batch in [batchA,batchB] :
    figures.append( plot.plot(batch, target='semantic', how='true', filter='none') )

ValueError: Plotly Express cannot process wide-form data with columns of different type.

### Save plots to disk

We can also use plotly's `write_html` and `write_image` methods to print the figure as an interactive webpage, or in a raster graphics (ie. PNG, JPEG etc) or vector graphics (ie. PDF) format. By default this writes to a `plots` subdirectory – if you're seeing an error that this directory does not exist, simply create one, or change the path to a valid output location!

In [ ]:
name=nameA
plot_dir="/home/twalton/NuGraphPlots/Events"
figA.write_html(f'{plot_dir}/{name}_semantic_true.html')
#figA.write_image(f'{plot_dir}/{name}_semantic_true.png')
#figA.write_image(f'{plot_dir}/{name}_semantic_true.pdf')

### (Optional) Select example events

The following blocks will select the representative events from the NuGraph2 paper

### Event 1

Run 5189, subrun 225, event 11300

In [ ]:
data = nudata.combined_test[0]
md = data['metadata']
name = f'r{md.run}_sr{md.subrun}_e{md.event}'
model.step(data);

### Event 2

Run 6999, subrun 11, event 595

In [ ]:
data = nudata.test_dataset[36]
md = data['metadata']
name = f'r{md.run}_sr{md.subrun}_e{md.event}'
model.step(data);

### Event 3

Run 7048, subrun 177, event 8858

In [ ]:
data = nudata.test_dataset[11]
md = data['metadata']
name = f'r{md.run}_sr{md.subrun}_e{md.event}'
model.step(data);

### Event 4

Run 5459, subrun 94, event 4738

In [ ]:
data = nudata.test_dataset[91]
md = data['metadata']
name = f'r{md.run}_sr{md.subrun}_e{md.event}'
model.step(data);

### Event 5

Run 6780, subrun 200, event 10006

In [ ]:
data = nudata.test_dataset[27]
md = data['metadata']
name = f'r{md.run}_sr{md.subrun}_e{md.event}'
model.step(data);

### Plot event displays

Write event displays to disk in PDF format for use in the NuGraph2 paper.

In [ ]:
plot.plot(data, target='filter', how='true', filter='none').write_image(f'plots/evd/{name}_filter_true.pdf')
plot.plot(data, target='filter', how='pred', filter='none').write_image(f'plots/evd/{name}_filter_pred.pdf')
plot.plot(data, target='semantic', how='true', filter='true').write_image(f'plots/evd/{name}_semantic_true.pdf')
plot.plot(data, target='semantic', how='pred', filter='pred').write_image(f'plots/evd/{name}_semantic_pred.pdf')

### Print model performance

Print out information on the rate at which the model makes mistakes, and some information on common failure modes.

In [ ]:
tf = torch.cat([(data[p].y_semantic!=-1) for p in nudata.planes])
pf = torch.cat([data[p].x_filter.round() for p in nudata.planes])
ts = torch.cat([data[p].y_semantic for p in nudata.planes])[tf]
ps = torch.cat([data[p].x_semantic.argmax(dim=1) for p in nudata.planes])[tf]

print(f'there are {tf.size(0)} hits overall, of which {tf.sum()} are signal.')

print('\n### Filter\n')

mask = tf != pf
print(f'{mask.sum()} hits were classified wrong. of those, {(tf[mask]==0).sum()} are false positives, and {(tf[mask]==1).sum()} are false negatives.')

print('\n### Semantic\n')

print(f'of the {tf.sum()} signal hits, {(ps==ts).sum()} are correctly classified.')

mask = ts != ps
print(f'of the {mask.sum()} misclassified hits:')

for i, c in enumerate(nudata.semantic_classes):
    tm = ts[mask]==i
    if tm.sum() == 0: continue
    print(f'- {tm.sum()} {c} hits were misclassified.')
    for j, cj in enumerate(nudata.semantic_classes):
        pm = ps[mask][tm]==j
        if pm.sum() == 0: continue
        print(f'  - {pm.sum()} as {cj}')